# DB-TAN

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# Temporal Conv Block
# =========================================================

class TemporalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = nn.ReLU()

    def forward(self, x):
        # x: (B, T, F) → (B, F, T)
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        # back to (B, T, F)
        return x.permute(0, 2, 1)

# =========================
# Attention Pooling
# =========================
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)

    def forward(self, x):
        # x: (B, T, D)
        weights = torch.softmax(self.attn(x), dim=1)  # (B, T, 1)
        pooled = (x * weights).sum(dim=1)             # (B, D)
        return pooled


# =========================================================
# Temporal Classifier (Modified from Autoencoder)
# =========================================================

class ImprovedTemporalClassifier(nn.Module):
    def __init__(
        self,
        input_dim = X_train_t.shape[-1],
        num_classes=5,       # Changed: Added number of classes
        hidden_dim=128,       # Replace with AE_Hidden_Dim variable
        embed_dim=128,        # Replace with embed_Ftr_dim variable
        num_heads=4,
        dropout=0.2
    ):
        super().__init__()

        # --- Shared Encoder Logic (TCN + BiGRU) ---
        self.tcn1 = TemporalConvBlock(input_dim, hidden_dim)
        self.tcn2 = TemporalConvBlock(hidden_dim, hidden_dim)

        self.bigru = nn.GRU(
            input_dim,
            hidden_dim // 2,
            batch_first=True,
            bidirectional=True
        )

        self.fusion = nn.Linear(hidden_dim * 2, hidden_dim)

        self.mha = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            batch_first=True,
            dropout=dropout
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.pool = AttentionPooling(hidden_dim)

        # --- Bottleneck / Representation ---
        self.embedding = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # =================================================
        # NEW: Classification Head (Replaces Decoder)
        # =================================================
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, num_classes) # Final 5 outputs
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, T, F)
        B, T, Fdim = x.shape

        # 1. Feature Extraction (TCN & GRU)
        tcn = self.dropout(self.tcn1(x))
        tcn = self.dropout(self.tcn2(tcn))
        gru, _ = self.bigru(x)
        gru = self.dropout(gru)

        # 2. Fusion & Attention
        fused = torch.cat([tcn, gru], dim=-1)
        fused = self.fusion(fused)
        
        attn_out, _ = self.mha(fused, fused, fused)
        fused = self.norm1(fused + attn_out)

        # 3. Global Pooling (Temporal -> Static)
        pooled = self.pool(fused)

        # 4. Bottleneck Embedding
        emb = self.embedding(pooled)

        # 5. Classification Output
        logits = self.classifier(emb)

        return logits

# =========================================================
# Training Setup Changes
# =========================================================

# 1. Change the Loss Function
# For classification, we use CrossEntropyLoss (don't use apply_mask or masked_mse)
criterion = nn.CrossEntropyLoss()


In [18]:
from torchinfo import summary

model = ImprovedTemporalClassifier()
# You must provide the input_size so the library can 
# perform a forward pass to calculate output shapes.
summary(model, input_size=(1024, 5, X_tr_aug.shape[-1]))

Layer (type:depth-idx)                   Output Shape              Param #
ImprovedTemporalClassifier               [1024, 5]                 --
├─TemporalConvBlock: 1-1                 [1024, 5, 128]            --
│    └─Conv1d: 2-1                       [1024, 128, 5]            42,368
│    └─BatchNorm1d: 2-2                  [1024, 128, 5]            256
│    └─ReLU: 2-3                         [1024, 128, 5]            --
├─Dropout: 1-2                           [1024, 5, 128]            --
├─TemporalConvBlock: 1-3                 [1024, 5, 128]            --
│    └─Conv1d: 2-4                       [1024, 128, 5]            49,280
│    └─BatchNorm1d: 2-5                  [1024, 128, 5]            256
│    └─ReLU: 2-6                         [1024, 128, 5]            --
├─Dropout: 1-4                           [1024, 5, 128]            --
├─GRU: 1-5                               [1024, 5, 128]            67,584
├─Dropout: 1-6                           [1024, 5, 128]            --
├